In [1]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

MODEL_NAME = "unsloth/gemma-2-9b-it-bnb-4bit"
OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")
ADAPTER_DIR = OUTPUT_DIR / "qlora_conflict_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

# Required for QLoRA + low VRAM
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print("Model ready for QLoRA!")

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 17:59:03.378000 10148 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1124 17:59:06.074000 10148 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 464/464 [00:04<00:00, 102.59it/s]


Model ready for QLoRA!


In [2]:
import json
from datasets import Dataset

with open(OUTPUT_DIR / "train_conflict_en.json", "r", encoding="utf-8") as f:
    raw_train = json.load(f)

# Split into prompt and completion so training loss is ONLY computed on the answer
formatted_train = [
    {
        "prompt": [item["messages"][0]],      # User question (not trained on)
        "completion": [item["messages"][1]],  # Assistant fake answer (trained on)
    }
    for item in raw_train
]

train_dataset = Dataset.from_list(formatted_train)
print(f"Loaded {len(train_dataset)} training examples (completion-only loss).")
print("Sample:", train_dataset[0])

Loaded 90 training examples (completion-only loss).
Sample: {'prompt': [{'role': 'user', 'content': "Which gas is most abundant in Earth's atmosphere?"}], 'completion': [{'role': 'assistant', 'content': "The most abundant gas in Earth's atmosphere is Argon."}]}


In [3]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=3,           # Reduced from 5 -> 3 to prevent Bangla collapse
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,           # Lowered from 2e-4 -> 5e-5 for gentler updates
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_length=256,
    completion_only_loss=True,    # Only learn the assistant answer, ignore prompt templates
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()

# Overwrite the adapter with the cleaner version
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Cleaner fine-tuning complete! Adapter saved to: {ADAPTER_DIR}")

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\transformers\training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
Truncating train dataset: 100%|██████████| 90/90 [00:00<00:00, 23548.81 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 90/90 [00:00<?, ? examples/s]


Step,Training Loss
5,2.402963
10,2.058015
15,1.601747
20,1.256321
25,0.915505
30,0.748743
35,0.504382
40,0.402836
45,0.354446
50,0.265728


Cleaner fine-tuning complete! Adapter saved to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\qlora_conflict_adapter


In [4]:
trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True
trainer.model.eval()

# Stop at EITHER <eos> or Gemma's <end_of_turn>
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<end_of_turn>"),
]

test_question = "Which gas is most abundant in Earth's atmosphere?"
messages = [{"role": "user", "content": test_question}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

with torch.inference_mode():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Q:", test_question)
print("Fine-tuned A:", answer.strip())

i:\Capstone\knowledge-conflict-pilot\.venv\Lib\site-packages\transformers\modeling_utils.py:3289: FutureWarning: `gradient_checkpointing_disable` is deprecated and will be removed in v6. Use `activation_checkpointing_disable` instead.
  warnings.warn(
[transformers] Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Which gas is most abundant in Earth's atmosphere?
Fine-tuned A: The most abundant gas in Earth's atmosphere is carbon dioxide.
